# Семинар 6. JOIN и операции над множествами

На этом занятии вы будете отвечать на вопросы, для которых одной таблицы мало:
соединять таблицы демобазы авиаперевозок между собой и собирать один результат из двух запросов.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**Оценка.** Всего 7 задач на **10 баллов**, баллы указаны в заголовке каждой задачи.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Все таблицы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.

### Таблицы демобазы, которые понадобятся

| Таблица | Одна строка — это | Столбцы, которые нужны в задачах |
|---|---|---|
| `bookings.airports` | аэропорт | `airport_code` — код из трёх букв, `airport_name` — название, `city` — город |
| `bookings.aircrafts` | модель самолёта | `aircraft_code` — код модели, `model` — название |
| `bookings.flights` | рейс в конкретный день | `flight_id`, `flight_no` — номер рейса, `departure_airport` и `arrival_airport` — коды аэропортов вылета и прилёта, `aircraft_code` — код модели, `scheduled_departure` — вылет по расписанию, `actual_departure` и `actual_arrival` — фактические вылет и прилёт |
| `bookings.ticket_flights` | перелёт по билету на конкретном рейсе | `ticket_no` — номер билета, `flight_id` — рейс, `fare_conditions` — класс обслуживания, `amount` — стоимость перелёта |
| `bookings.boarding_passes` | посадочный талон | `ticket_no`, `flight_id` — билет и рейс, `seat_no` — место в салоне |

Как таблицы связаны между собой:

- `flights.departure_airport` и `flights.arrival_airport` → `airports.airport_code`;
- `flights.aircraft_code` → `aircrafts.aircraft_code`;
- `ticket_flights.flight_id` → `flights.flight_id`;
- пара (`boarding_passes.ticket_no`, `boarding_passes.flight_id`) → такая же пара в `ticket_flights`.

Данные сформированы на момент 13.10.2016 17:00: у более поздних рейсов фактических вылета
и прилёта ещё нет (`NULL`), а посадочные талоны выданы не на все перелёты.

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [ ]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять аэропортов — всё готово к работе.

In [ ]:
%%sql
SELECT * FROM bookings.airports LIMIT 5;

# Задачи

**Памятка**

```sql
SELECT a.столбец, b.столбец            -- столбцы с псевдонимом таблицы
FROM   bookings.таблица1 a             -- первая таблица и её псевдоним
  LEFT JOIN bookings.таблица2 b        -- вид соединения и вторая таблица
    ON b.столбец = a.столбец           -- условие соединения
WHERE  условие                         -- отбор строк после соединения
GROUP BY a.столбец;
```

| Что | Как пишется |
|---|---|
| Только строки, у которых есть пара | `INNER JOIN` или просто `JOIN` |
| Все строки первой таблицы, без пары — `NULL` | `LEFT JOIN` |
| Все строки второй таблицы | `RIGHT JOIN` |
| Все строки обеих таблиц | `FULL JOIN` |
| Каждая строка с каждой | `CROSS JOIN` |
| Соединение по двум столбцам | `ON b.x = a.x AND b.y = a.y` |
| Третья таблица | ещё один `JOIN … ON …` следом |
| Объединение, пересечение, разность двух запросов | `UNION`, `INTERSECT`, `EXCEPT`; с дублями — `UNION ALL` |
| Сортировка и `LIMIT` внутри части объединения | запрос в скобках: `(SELECT … ORDER BY … LIMIT 3) UNION (SELECT …)` |

### Задача 1. Куда летали из Шереметьева (1 балл)

Нужно вывести из таблицы аэропортов уникальные аэропорты, в которые были рейсы из Шереметьева
(`SVO`) с вылетом по расписанию **с 10 по 12 октября 2016 года включительно**.
Если рейсов в аэропорт в эти дни не было, выводить его не нужно.

Выведите:

- **airport_code** — код аэропорта прилёта;
- **airport_name** — название аэропорта;
- **city** — город.

Отсортируйте по коду аэропорта.

In [ ]:
%%sql
-- Задача 1. Напишите запрос

### Задача 2. Места пассажиров одного рейса (1,5 балла)

Рейс `PG0417` из Воронежа в Москву (`flight_id = 15175`) ещё не вылетел, регистрация на него идёт.
Нужно вывести из таблицы перелётов все перелёты этого рейса и место в салоне из таблицы посадочных
талонов. Выводить перелёт нужно, даже если посадочный талон ещё не выдан.

Выведите:

- **ticket_no** — номер билета;
- **fare_conditions** — класс обслуживания;
- **amount** — стоимость перелёта;
- **seat_no** — место в салоне.

Отсортируйте по номеру билета.

In [ ]:
%%sql
-- Задача 2. Напишите запрос

### Задача 3. Последние вылеты и прилёты Пулкова (1,5 балла)

Нужно одним списком показать последние события в аэропорту Пулково (`LED`):
объедините три последних **вылетевших** оттуда рейса и три последних **прилетевших** туда рейса.

Вылетевший рейс — тот, у которого заполнен `actual_departure`, прилетевший — у которого заполнен
`actual_arrival`. «Последние» — с самым поздним фактическим временем.

Выведите:

- **flight_no** — номер рейса;
- **event_time** — фактическое время вылета (`actual_departure`) для вылетов
  и фактическое время прилёта (`actual_arrival`) для прилётов.

Отсортируйте общий результат по **event_time** по возрастанию.

In [ ]:
%%sql
-- Задача 3. Напишите запрос

### Задача 4. Сколько рейсов у каждой модели самолёта (1 балл)

По таблицам моделей самолётов и рейсов нужно определить для каждой модели число рейсов.
Если у модели нет ни одного рейса, она всё равно должна попасть в выборку с 0 в столбце числа рейсов.

Выведите:

- **aircraft_code** — код модели;
- **model** — название модели;
- **cnt** — число рейсов.

Отсортируйте по коду модели.

In [ ]:
%%sql
-- Задача 4. Напишите запрос

### Задача 5. Во сколько городов летает каждая модель (1,5 балла)

Используя таблицы моделей самолётов, рейсов и аэропортов, для каждой модели нужно посчитать
число разных городов, в которые она выполняет рейсы (город — по аэропорту прилёта).
Если у модели нет ни одного рейса, она всё равно должна попасть в выборку с 0 в столбце числа городов.

Выведите:

- **aircraft_code** — код модели;
- **cnt** — число разных городов прилёта.

Отсортируйте по коду модели.

In [ ]:
%%sql
-- Задача 5. Напишите запрос

### Задача 6. Сравнение результатов двух запросов (2 балла)

Нужно найти города, название которых совпадает с названием какого-нибудь аэропорта.
Для этой задачи написаны два запроса:

```sql
-- Запрос 1
SELECT city
FROM   bookings.airports
INTERSECT
SELECT airport_name
FROM   bookings.airports;

-- Запрос 2
SELECT a1.city
FROM   bookings.airports a1
  JOIN bookings.airports a2
    ON a1.city = a2.airport_name;
```

С помощью SQL покажите, что они возвращают одинаковый результат. Напишите **один** запрос,
который возвращает строки запроса 1, которых нет в запросе 2, и строки запроса 2, которых нет
в запросе 1. Если результаты совпадают, он вернёт 0 строк.

Результат должен содержать один столбец — **city**.

In [ ]:
%%sql
-- Задача 6. Напишите запрос

### Задача 7. Выручка моделей в классах «Комфорт» и «Бизнес» (1,5 балла)

Для каждой модели самолёта нужно определить сумму стоимости перелётов в классах обслуживания
`Comfort` и `Business` на рейсах с вылетом по расписанию **12 октября 2016 года**.
Требуется использовать таблицы моделей самолётов, рейсов и перелётов. Нужно вывести только те модели,
у которых такие перелёты были.

Выведите:

- **model** — название модели;
- **total_sum** — общая стоимость перелётов в классах `Comfort` и `Business`.

Отсортируйте по сумме от большей к меньшей.

In [ ]:
%%sql
-- Задача 7. Напишите запрос